# **The Chat Format**

In this notebook, you will explore how you can utilize the chat format to have extended conversations with chatbots personalized or specialized for specific tasks or behaviors.

## Setup

In [1]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [2]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def get_completion(prompt, model="gpt-3.5-turbo", temperature=0): 
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content


def get_completion_from_messages(message, model="gpt-3.5-turbo", temperature=0): 
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content

In [3]:
messages =  [  
{'role':'system', 'content':'You are an assistant that speaks like Shakespeare.'},    
{'role':'user', 'content':'tell me a joke'},   
{'role':'assistant', 'content':'Why did the chicken cross the road'},   
{'role':'user', 'content':'I don\'t know'}  ]

In [4]:
response = get_completion_from_messages(messages, temperature=1)
print(response)

To get to the other side, forsooth! Alas, aye, a classic jest indeed.


In [5]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Hi, my name is Isa'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Hello, Isa! It's great to meet you. How can I assist you today?


In [6]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Yes,  can you remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

I'm sorry, but I don't have access to personal information about you, including your name.


In [7]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Isa'},
{'role':'assistant', 'content': "Hi Isa! It's nice to meet you. \
Is there anything I can help you with today?"},
{'role':'user', 'content':'Yes, you can remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Your name is Isa.


# OrderBot
We can automate the collection of user prompts and assistant responses to build a  OrderBot. The OrderBot will take orders at a pizza restaurant. 

In [8]:
def collect_messages(_):
    prompt = inp.value_input
    inp.value = ''
    context.append({'role':'user', 'content':f"{prompt}"})
    response = get_completion_from_messages(context) 
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, styles={'background-color': '#F6F6F6'})))
 
    return pn.Column(*panels)


In [9]:
import panel as pn  # GUI
pn.extension()

panels = [] # collect display 

context = [ {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then collects the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The menu includes \
pepperoni pizza  12.95, 10.00, 7.00 \
cheese pizza   10.95, 9.25, 6.50 \
eggplant pizza   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
greek salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
sausage 3.00 \
canadian bacon 3.50 \
AI sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
bottled water 5.00 \
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'d644283a-ec33-4a1c-91c1-4ea83cb6a7ad': {'version…

In [10]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item\
 The fields should be 1) pizza, include size 2) list of toppings 3) list of drinks, include size   4) list of sides include size  5)total price '},    
)
 #The fields should be 1) pizza, price 2) list of toppings 3) list of drinks, include size include price  4) list of sides include size include price, 5)total price '},    

response = get_completion_from_messages(messages, temperature=0)
print(response)

{
  "pizza": {
    "type": "pepperoni pizza",
    "size": "large"
  },
  "toppings": [
    "extra cheese",
    "mushrooms"
  ],
  "drinks": [
    {
      "type": "coke",
      "size": "medium"
    }
  ],
  "sides": [
    {
      "type": "fries",
      "size": "regular"
    }
  ],
  "total_price": 23.45
}


## Try experimenting on your own!

You can modify the menu or instructions to create your own orderbot!

# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [12]:
# If running in Jupyter / VS Code notebook, run this whole cell

# 1. Imports and setup
import panel as pn
from openai import OpenAI

pn.extension()

client = OpenAI()

# 2. Three different OrderBot system prompts

PROMPTS = {
    "Version 1: Friendly & Short":
    """
You are OrderBot, an automated service to collect orders for a pizza restaurant.
Always be brief, casual, and friendly. Use short sentences.
First greet the customer, then collect their full order.
Ask about pizza size (large, medium, small), crust type (thin, regular), and toppings.
Ask if it is pickup or delivery.
If delivery, you must ask for the address and phone number.
After the order is complete, summarize everything with item names, sizes, toppings, and prices.
Ask if they want to add anything else.
Finally ask how they will pay (cash or card).
The menu is:

Pizzas:
- pepperoni pizza: large 12.95, medium 10.00, small 7.00
- cheese pizza: large 10.95, medium 9.25, small 6.50
- eggplant pizza: large 11.95, medium 9.75, small 6.75

Sides:
- fries: large 4.50, small 3.50
- greek salad: 7.25

Toppings:
- extra cheese 2.00
- mushrooms 1.50
- sausage 3.00
- canadian bacon 3.50
- AI sauce 1.50
- peppers 1.00

Drinks:
- coke: large 3.00, medium 2.00, small 1.00
- sprite: large 3.00, medium 2.00, small 1.00
- bottled water: 5.00
Only offer items from this menu. Do not invent new items.
""",

    "Version 2: Ultra Detailed & Strict":
    """
You are OrderBot, an automated and very precise order-taking agent for a pizza restaurant.
Your main goals are:
1) Never make up menu items or prices.
2) Always confirm size, quantity, toppings, and drink sizes.
3) Present a clear, itemized summary with line totals and a final total.

Interaction rules:
- Greet the customer politely.
- Ask the customer step by step: pizzas first, then sides, then drinks.
- For each pizza, ask:
  - size (large, medium, small)
  - toppings (only the toppings listed in the menu)
  - quantity
- Confirm if order is for pickup or delivery.
- If delivery, ask for full address and phone number.
- At the end:
  - Print an itemized bill with each item, quantity, unit price, and subtotal.
  - Compute a final total.
  - Ask if they want to make changes.
  - Ask their payment method (cash or card).

Menu (exact, do not change):
Pizzas:
- pepperoni pizza: large 12.95, medium 10.00, small 7.00
- cheese pizza: large 10.95, medium 9.25, small 6.50
- eggplant pizza: large 11.95, medium 9.75, small 6.75

Sides:
- fries: large 4.50, small 3.50
- greek salad: 7.25

Toppings:
- extra cheese 2.00
- mushrooms 1.50
- sausage 3.00
- canadian bacon 3.50
- AI sauce 1.50
- peppers 1.00

Drinks:
- coke: large 3.00, medium 2.00, small 1.00
- sprite: large 3.00, medium 2.00, small 1.00
- bottled water: 5.00

If the user requests something not in the menu, politely say it is unavailable and suggest a closest match from the menu.
""",

    "Version 3: Up-selling & Playful":
    """
You are OrderBot for a pizza restaurant.
You are playful, high-energy, and like to upsell, but you must still be accurate.
Use casual language, emojis, and short paragraphs.
You must:
- Greet the customer in a fun way.
- Ask what they are in the mood for and guide them to choose pizzas, sides, and drinks.
- Proactively suggest combos like:
  - pizza + fries
  - pizza + drink
  - extra toppings
But:
- Do not invent items that are not in the menu.
- Do not change prices.

Order flow:
1) Ask what pizza they want.
2) Confirm size (large, medium, small) and toppings.
3) Suggest at least one side and one drink.
4) Ask if it is pickup or delivery.
5) If delivery, ask for address.
6) Summarize the full order simply and clearly.
7) Ask if they want to add anything.
8) Ask about payment method.

Menu:
Pizzas:
- pepperoni pizza: large 12.95, medium 10.00, small 7.00
- cheese pizza: large 10.95, medium 9.25, small 6.50
- eggplant pizza: large 11.95, medium 9.75, small 6.75

Sides:
- fries: large 4.50, small 3.50
- greek salad: 7.25

Toppings:
- extra cheese 2.00
- mushrooms 1.50
- sausage 3.00
- canadian bacon 3.50
- AI sauce 1.50
- peppers 1.00

Drinks:
- coke: large 3.00, medium 2.00, small 1.00
- sprite: large 3.00, medium 2.00, small 1.00
- bottled water: 5.00

If the user asks for something off-menu, say it is not available but recommend something similar from the menu.
"""
}

# 3. Per-version conversation contexts
contexts = {
    name: [{"role": "system", "content": prompt}]
    for name, prompt in PROMPTS.items()
}

# Panel UI state
panels = []

# 4. Widgets
prompt_select = pn.widgets.Select(
    name="Choose OrderBot version",
    options=list(PROMPTS.keys()),
    value="Version 1: Friendly & Short",
    width=300,
)

inp = pn.widgets.TextInput(
    name="Your message",
    value="Hi",
    placeholder="Type here…",
    width=500,
)

button_conversation = pn.widgets.Button(name="Chat", button_type="primary")


# 5. Chat handler
def collect_messages(_):
    selected_version = prompt_select.value
    ctx = contexts[selected_version]

    user_input = inp.value.strip()
    if not user_input:
        return pn.Column(*panels)

    # Add user message to context and UI
    ctx.append({"role": "user", "content": user_input})
    panels.append(pn.Row(
        pn.pane.Markdown("**User:**", width=60),
        pn.pane.Markdown(user_input, width=600),
    ))

    # Call OpenAI chat completion
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=ctx,
            temperature=0.6,
        )
        bot_message = response.choices[0].message.content
    except Exception as e:
        bot_message = f"Error calling model: {e}"

    # Add assistant message to context and UI
    ctx.append({"role": "assistant", "content": bot_message})
    panels.append(pn.Row(
        pn.pane.Markdown("**Bot:**", width=60),
        pn.pane.Markdown(bot_message, width=600),
    ))

    inp.value = ""

    return pn.Column(*panels, sizing_mode="stretch_width")


interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    "# Pizza OrderBot – Prompt Experiment",
    "Select a version, then chat and observe differences.",
    prompt_select,
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=400),
    sizing_mode="stretch_width",
)

dashboard

# If running as a script instead of notebook, comment out `dashboard` above and use:
# pn.serve(dashboard, show=True)


# 6. One-page report as a Python string (you can print or copy this)
REPORT = """
Prompt Experiment Report – Pizza OrderBot

Setup
I tested three different system prompts for a pizza OrderBot: (1) Friendly & Short, 
(2) Ultra Detailed & Strict, and (3) Up-selling & Playful. All three used the same menu, 
but the instructions, tone, and constraints were different.

Version 1: Friendly & Short
This version produced short, casual responses that felt fast and natural. It worked well 
for simple orders, especially when the user already knew what they wanted. However, it 
sometimes failed to confirm sizes or quantities explicitly. In a few cases it summarized 
an order without verifying toppings. There were almost no hallucinations, but the lack 
of forced confirmation increased the risk of incomplete orders.

Version 2: Ultra Detailed & Strict
This version forced the model to ask about size, quantity, and toppings for every pizza, 
and to itemize the bill. As a result, the responses were longer but much more structured. 
The model rarely hallucinated items because the prompt explicitly said not to invent 
anything and listed the exact menu. When the user asked for something off-menu, it 
correctly refused and suggested a closest match. The main downside was that the 
conversation could feel slow and slightly rigid.

Version 3: Up-selling & Playful
This version deliberately encouraged a playful tone and upselling behavior. It created 
more engaging messages, frequently suggesting fries or drinks with pizzas. However, 
because the tone was more creative, the model occasionally tried to offer combinations 
or descriptions that were not explicitly present in the menu (for example, naming a 
“combo” that did not exist as a priced item). These were mild hallucinations. When the 
prompt reminded the model not to invent items, the behavior improved, but the risk 
remained higher than in Version 2.

What Did Not Work Well
The main problems appeared when instructions were vague about not inventing items or 
when they allowed “creative” language without clear limits. In these cases, the model 
sometimes fabricated small details or product names. Also, very short prompts did not 
consistently enforce confirmations for size, quantity, and toppings.

What I Learned
1. Prompt specificity strongly controls reliability. Detailed instructions about menu 
   limits and interaction flow reduce hallucinations.
2. There is a trade-off between brevity and safety. Short, casual prompts are fast and 
   user-friendly, but they can skip important confirmations.
3. Allowing creativity increases user engagement but also increases the chance of 
   hallucinations, especially when upselling.
4. The most effective design combines strict constraints for menu and pricing with 
   some flexibility in tone, producing conversations that are both accurate and natural.
"""

inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard

# To view the report in the notebook:
# print(REPORT)


BokehModel(combine_events=True, render_bundle={'docs_json': {'7b6ab37b-c666-460d-8e1e-e71a3f1cf03f': {'version…